In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


from xgboost import XGBRegressor

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline


In [2]:
df = pd.read_csv("processed_dataset.csv")

In [3]:
## SVM

In [4]:
# ============================================
# SVM REGRESSION - PREDICTING SCORE
# ============================================

# ============================================
# FEATURES & TARGET
# ============================================

features = [
    "DRIVER_POINTS_BEFORE_RACE",
    "LAPS",
    "MILLISECONDS",
    "WEATHER_rain",
    "WEATHER_WET",
    "WEATHER_cloudy",
    "OVERTAKEN_POSITIONS_TOTAL",
    "DNF_COUNT",
    "LAPMEAN",
    "FASTESTLAP",
    "PS_COUNT",
    "SC_COUNT",
    "ROLLING_POINTS",
    "ROLLING_LAP",
    "LAP_CONSISTENCY",
    "RACE_INTERRUPTIONS",
    "OVERTAKE_RATIO",
    "DRIVER_ENCODED",
    "RACE_ENCODED"
]

target = "SCORE"

# ============================================
# CREATE CLEAN DATAFRAME
# ============================================

model_df = df[features + [target]].copy()

# replace inf
model_df = model_df.replace([np.inf, -np.inf], np.nan)

# remove missing rows
model_df = model_df.dropna()

# ============================================
# X AND y
# ============================================

X = model_df[features]

# make y 1D
y = model_df[target].values.ravel()



# ============================================
# TRAIN / TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ============================================
# SVM PIPELINE
# ============================================

# IMPORTANT:
# SVM is VERY sensitive to scaling,
# so StandardScaler is essential.

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(
        kernel="rbf",   # most common kernel
        C=100,
        gamma="scale",
        epsilon=0.1
    ))
])

# ============================================
# TRAIN MODEL
# ============================================

svm_model.fit(X_train, y_train)

# ============================================
# PREDICTIONS
# ============================================

y_pred = svm_model.predict(X_test)

# ============================================
# EVALUATION
# ============================================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("========== SVM RESULTS ==========")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

# ============================================
# SAMPLE PREDICTIONS
# ============================================

results = pd.DataFrame({
    "Actual_SCORE": y_test,
    "Predicted_SCORE": y_pred
})

print("\nSample predictions:")
print(results.head(10))

# ============================================
# OPTIONAL: SAVE MODEL
# ============================================

import joblib

joblib.dump(svm_model, "svm_score_model.pkl")

print("\nModel saved as svm_score_model.pkl")

========== SVM RESULTS ==========
MAE  : 0.8229
RMSE : 0.9836
R2   : 0.4969

Sample predictions:
   Actual_SCORE  Predicted_SCORE
0      4.270024         6.270521
1      9.100000         8.221197
2      5.396000         5.756387
3      8.200000         8.253770
4      7.860000         7.940639
5      7.662000         8.874707
6      8.370000         7.750488
7      5.440000         5.803278
8      4.500000         4.815354
9      5.630000         5.273229

Model saved as svm_score_model.pkl


In [5]:
## XGBOOST

In [6]:
# ============================================
# XGBOOST REGRESSION - PREDICTING SCORE
# ============================================

# ============================================
# CLEAN TARGET
# ============================================
# ============================================
# CLEAN DATA FOR XGBOOST
# ============================================

# Crear dataframe solo con features y target
model_df = df[features + [target]].copy()

# Convertir TODO a numérico
for col in model_df.columns:
    model_df[col] = pd.to_numeric(
        model_df[col],
        errors="coerce"
    )

# Reemplazar infinitos
model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Ver cuántos NaNs hay
print(model_df.isna().sum())

# Eliminar filas problemáticas
model_df = model_df.dropna()


# ============================================
# X AND y
# ============================================

X = model_df[features]
y = model_df[target]


# ============================================
# TRAIN / TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ============================================
# XGBOOST MODEL
# ============================================

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

# ============================================
# TRAIN MODEL
# ============================================

xgb_model.fit(X_train, y_train)

# ============================================
# PREDICTIONS
# ============================================

y_pred = xgb_model.predict(X_test)

# ============================================
# EVALUATION
# ============================================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("========== XGBOOST RESULTS ==========")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

# ============================================
# FEATURE IMPORTANCE
# ============================================

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop Feature Importances:")
print(importance_df)

# ============================================
# SAMPLE PREDICTIONS
# ============================================

results = pd.DataFrame({
    "Actual_SCORE": y_test.values,
    "Predicted_SCORE": y_pred
})

print("\nSample predictions:")
print(results.head(10))

# ============================================
# OPTIONAL: SAVE MODEL
# ============================================

import joblib

joblib.dump(xgb_model, "xgboost_score_model.pkl")

print("\nModel saved as xgboost_score_model.pkl")

DRIVER_POINTS_BEFORE_RACE        0
LAPS                          8601
MILLISECONDS                  8601
WEATHER_rain                   350
WEATHER_WET                    350
WEATHER_cloudy                 350
OVERTAKEN_POSITIONS_TOTAL      350
DNF_COUNT                      350
LAPMEAN                      33014
FASTESTLAP                   33014
PS_COUNT                     29188
SC_COUNT                     30591
ROLLING_POINTS                   0
ROLLING_LAP                  32997
LAP_CONSISTENCY              33059
RACE_INTERRUPTIONS           31244
OVERTAKE_RATIO               11308
DRIVER_ENCODED                   0
RACE_ENCODED                     0
SCORE                          350
dtype: int64
========== XGBOOST RESULTS ==========
MAE  : 0.5941
RMSE : 0.7432
R2   : 0.7128

Top Feature Importances:
                      Feature  Importance
10                   PS_COUNT    0.179877
16             OVERTAKE_RATIO    0.179438
15         RACE_INTERRUPTIONS    0.163545
6   OVERTAKEN